<a href="https://colab.research.google.com/github/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/blob/main/simple_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone 'https://github.com/vishaljoshi24/Dungeons-and-Dragons-Turn-Classification/'

Cloning into 'Dungeons-and-Dragons-Turn-Classification'...
remote: Enumerating objects: 431, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (27/27), done.
remote: Total 431 (delta 70), reused 56 (delta 56), pack-reused 348 (from 2)
Receiving objects: 100% (431/431), 3.07 MiB | 19.28 MiB/s, done.
Resolving deltas: 100% (250/250), done.


In [2]:
!pip install dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 25.2 MB/s eta 0:00:00
  Attempting uninstall: typeguard
    Found existing installation: typeguard 4.5.2
    Uninstalling typeguard-4.5.2:
      Successfully uninstalled typeguard-4.5.2


In [3]:
import dspy

In [4]:
lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434', api_key='', max_tokens=2048)
dspy.configure(lm=lm)

In [6]:
import pandas as pd

instructions_df = pd.read_excel('instructions.xlsx')

In [7]:
instructions_df

,Unnamed: 0,0,1
0,0,Worldbuilding,"When engaging in Worldbuilding, actively contr..."
1,1,Worldbuilding,"When engaging in Worldbuilding, actively contr..."
2,2,Worldbuilding,"When engaging in Worldbuilding, actively contr..."
3,3,Worldbuilding,"When engaging in Worldbuilding, actively contr..."
4,4,Strategising,"When strategising, consider the flow of the ga..."
5,5,Gameplay Mechanic,"Follow the established rules for combat, skill..."
6,6,Gameplay Mechanic,"Follow the established rules for combat, skill..."
7,7,Gameplay Mechanic,"Follow the established rules for combat, skill..."
8,8,Gameplay Mechanic,"Follow the established rules for combat, skill..."
9,9,Gameplay Mechanic,"Follow the established rules for combat, skill..."


In [8]:
prompts_df = instructions_df.drop_duplicates(subset=[0], inplace=False)

In [9]:
prompts_df

,Unnamed: 0,0,1
0,0,Worldbuilding,"When engaging in Worldbuilding, actively contr..."
4,4,Strategising,"When strategising, consider the flow of the ga..."
5,5,Gameplay Mechanic,"Follow the established rules for combat, skill..."
11,11,Out-Of-Game Conversation,"When engaging in out-of-game conversations, sp..."


In [10]:
prompts_df = prompts_df.drop(columns=['Unnamed: 0'], inplace=False)

In [11]:
prompts_df

,0,1
0,Worldbuilding,"When engaging in Worldbuilding, actively contr..."
4,Strategising,"When strategising, consider the flow of the ga..."
5,Gameplay Mechanic,"Follow the established rules for combat, skill..."
11,Out-Of-Game Conversation,"When engaging in out-of-game conversations, sp..."


In [12]:
prompts = []

for classes, instructions in prompts_df.values:
    examples = dspy.Example(classes=classes, instructions=instructions).with_inputs("classes", "instructions")
    prompts.append(examples)

In [13]:
prompts

[Example({'classes': 'Worldbuilding', 'instructions': "When engaging in Worldbuilding, actively contribute to shaping the game world by sharing ideas about your character's background, the region they hail from, or unique cultural practices. Ask thoughtful questions about the world's history or magic to deepen the narrative. Collaborate with your DM to ensure your actions align with the world's rules and lore, making the setting feel dynamic and integral to your story."}) (input_keys={'classes', 'instructions'}),
 Example({'classes': 'Strategising', 'instructions': "When strategising, consider the flow of the game, allocate resources (e.g., spells, actions) wisely, communicate with your party to align goals, and adapt your approach based on emerging threats or opportunities. Balance short-term gains with long-term objectives to maintain flexibility and maximize your group's effectiveness."}) (input_keys={'classes', 'instructions'}),
 Example({'classes': 'Gameplay Mechanic', 'instructio

In [15]:
import pandas as pd
import datetime

class PromptTool:
  """Tool for retrieving appropriate instruction from excel spreadsheet."""
  def __init__(self):
    self.prompts = prompts

  def search_prompts(self, query: str, user_id: str = "default_user", limit: int=4) -> str:
    """Search for the relevant instruction prompt given a user's input"""
    try:
      prompt = self.prompts.search(query, user_id=user_id, limit=limit)
      if not prompt:
        return "No relevant prompt found"

      prompt_text = "Using the following promt:\n"
      for i, prompt in enumerate(prompts["instructions"]):
        prompt_text += f"{i}.{prompt['instructions']}\n"
        return prompt_text
    except Exception as e:
      return f"Error searching prompts: {str(e)}"

  def update_prompt(self, prompt_class: str, new_prompt: str) -> str:
    """Update the existing instruction prompt if appropriate to do so."""
    try:
      self.prompt.update(prompt_class, new_prompt)
      return f"Updated prompt with new instructions: {new_prompt}"
    except Exception as e:
      return f"Error updating instructions: {str(e)}"

def get_current_time() -> str:
    """Get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [17]:
class ChatbotSignature(dspy.Signature):
  """
  You are a Dungeons & Dragons player agent and have access to a spreadsheet of
  instruction prompts paired with a category of player behaviour. Whenever you
  answer a user's input, respond according to the instructions which best match
  the situation the user's input creates.
  """
  user_input: str = dspy.InputField()
  response: str = dspy.OutputField()

class InstructedChatbot(dspy.Module):
  """A ReAct agent enhanced with a set of instructions based on Dungeons & Dragons
  player behaviour"""

  def __init__(self):
    super().__init__()
    self.prompt_tools = PromptTool

    self.tools = [
        self.prompt_tools.search_prompts,
        self.prompt_tools.update_prompt,
    ]

    self.react = dspy.ReAct(
        signature=ChatbotSignature,
        tools = self.tools,
        max_iters=6
    )

  def forward(self, user_input: str):
    """Process user input with prompt-aware reasoning"""
    return self.react(user_input=user_input)



In [ ]:
def run_agent_demo():
  """Demonstration of prompt-enhanced ReAct agent."""
  lm = dspy.LM('ollama_chat/qwen3:8b', api_base = 'http://localhost:11434',
               api_key='', max_tokens=2048)
  dspy.configure(lm=lm)

  agent = InstructedChatbot()

  print("Prompt Enhanced ReAct Agent Demo")
  print("="*50)

  user_input = Input
